# CLIP + FAISS Image Search

Text-to-image search over 500 CIFAR-10 images using OpenAI CLIP embeddings, FAISS nearest-neighbor lookup, and a Gradio UI.

## 1. Setup

In [ ]:
!pip -q install transformers faiss-cpu gradio

In [ ]:
import os
import pickle
import numpy as np
import torch
import faiss
import gradio as gr
import matplotlib.pyplot as plt

from PIL import Image
from torchvision.datasets import CIFAR10
from transformers import CLIPProcessor, CLIPModel

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 2. Data prep

Download CIFAR-10 test split and save the first 500 images as PNGs into `project_data/images/`.

In [ ]:
os.makedirs("project_data/images", exist_ok=True)
print("Folder project_data/images is ready.")

In [ ]:
dataset = CIFAR10(root="project_data", train=False, download=True)
print("CIFAR-10 downloaded.")
print("Number of images in test split:", len(dataset))

In [ ]:
max_images = 500

for i in range(max_images):
    img, label = dataset[i]
    img.save(f"project_data/images/img_{i:04d}.png")

print(f"Saved {max_images} images into project_data/images")

In [ ]:
sample_path = "project_data/images/img_0000.png"
sample_img = Image.open(sample_path)

plt.imshow(sample_img)
plt.axis("off")
plt.title("Sample CIFAR Image")
plt.show()

## 3. Model load

Load pretrained `openai/clip-vit-base-patch32` via `CLIPProcessor` / `CLIPModel`.

In [ ]:
model_name = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(model_name)
model = CLIPModel.from_pretrained(model_name)

model = model.to(device)
model.eval()

print("CLIP model loaded")

## 4. Image embeddings

Run CLIP's `get_image_features` on each PNG and L2-normalize so inner product equals cosine similarity.

In [ ]:
image_folder = "project_data/images"

image_paths = []
for file in os.listdir(image_folder):
    if file.endswith(".png"):
        image_paths.append(os.path.join(image_folder, file))

image_paths = sorted(image_paths)
print("Total images:", len(image_paths))

In [ ]:
def get_image_embedding(img_path):
    img = Image.open(img_path).convert("RGB")

    inputs = processor(images=img, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        features = model.get_text_features(**inputs)

    features = features / torch.norm(features, dim=-1, keepdim=True)
    return features.cpu().numpy()

## 5. Persistence

Cache `image_embeddings.npy` and `image_paths.pkl` so later runs skip the 500-image encode pass, then build a FAISS `IndexFlatIP` on the normalized vectors and save it to `faiss_index.bin`.

In [ ]:
embeddings_path = "project_data/image_embeddings.npy"
paths_path = "project_data/image_paths.pkl"
index_path = "project_data/faiss_index.bin"

if os.path.exists(embeddings_path) and os.path.exists(paths_path):
    image_embeddings = np.load(embeddings_path)
    with open(paths_path, "rb") as f:
        image_paths = pickle.load(f)
    print("Loaded cached embeddings:", image_embeddings.shape)
else:
    all_embeddings = []
    for i, path in enumerate(image_paths):
        emb = get_image_embedding(path)
        all_embeddings.append(emb[0])
        if (i + 1) % 50 == 0:
            print("Processed", i + 1, "images")

    image_embeddings = np.array(all_embeddings).astype("float32")

    np.save(embeddings_path, image_embeddings)
    with open(paths_path, "wb") as f:
        pickle.dump(image_paths, f)

    print("finished creating embeddings")
    print("Shape:", image_embeddings.shape)

In [ ]:
if os.path.exists(index_path):
    index = faiss.read_index(index_path)
    print("Loaded FAISS index from disk. ntotal =", index.ntotal)
else:
    dim = image_embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(image_embeddings)
    faiss.write_index(index, index_path)
    print("Built FAISS IndexFlatIP. ntotal =", index.ntotal)

## 6. Text query

Mirror the image encoder with `model.get_text_features` and the same L2 normalization.

In [ ]:
def get_text_embedding(text):
    inputs = processor(text=[text], return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        features = model.get_image_features(**inputs)

    features = features / torch.norm(features, dim=-1, keepdim=True)
    return features.cpu().numpy().astype("float32")

## 7. Search

Encode the query text, run FAISS `index.search`, return the top-k paths and scores.

In [ ]:
def search(query, k=5):
    vec = get_text_embedding(query)
    scores, idxs = index.search(vec, k)
    paths = [image_paths[i] for i in idxs[0]]
    return paths, scores[0].tolist()

In [ ]:
paths, scores = search("a red car", k=5)
for p, s in zip(paths, scores):
    print(f"{s:.4f}  {p}")

## 8. Gradio UI

Textbox for the query, slider for `k`, gallery of retrieved images with similarity scores.

In [ ]:
def gradio_search(query, k):
    if not query or not query.strip():
        return []
    paths, scores = search(query, k=int(k))
    return [(p, f"score: {s:.4f}") for p, s in zip(paths, scores)]

demo = gr.Interface(
    fn=gradio_search,
    inputs=[
        gr.Textbox(label="Query", placeholder="e.g. 'a red car' or 'dog on grass'"),
        gr.Slider(minimum=1, maximum=20, step=1, value=5, label="k"),
    ],
    outputs=gr.Gallery(label="Results", columns=5, height="auto"),
    title="CLIP + FAISS Image Search",
    description="Text-to-image retrieval over 500 CIFAR-10 images.",
)

demo.launch()